# REGEX

# Spacy

In [ ]:
import sys
!{sys.executable} -m pip install spacy pandas
!{sys.executable} -m spacy download en_core_web_sm


Defaulting to user installation because normal site-packages is not writeable
  Using cached spacy-3.8.17.tar.gz (1.3 MB)
  distutils: /private/var/folders/pq/t2g6jyg53hnf83397f72gtw80000gn/T/pip-build-env-hk5muyh5/normal/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  distutils: /private/var/folders/pq/t2g6jyg53hnf83397f72gtw80000gn/T/pip-build-env-hk5muyh5/normal/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  user = False
  home = None
  root = None
  prefix = '/private/var/folders/pq/t2g6jyg53hnf83397f72gtw80000gn/T/pip-build-env-hk5muyh5/normal'
  distutils: /private/var/folders/pq/t2g6jyg53hnf83397f72gtw80000gn/T/pip-build-env-hk5muyh5/overlay/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  distutils: /private/var/folders/pq/t2g6jyg53hnf83397f72gtw80000gn/T/pip-build-env-hk5muyh5/overlay/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  user = False
  home = None
  ro

In [ ]:
# =============================================================================
# ANONYMISATION DU DATASET AVEC spaCy
# - Détecte les données personnelles dans comment_text
# - Les remplace par une étiquette : "John" -> "_PERSON_", "1.2.3.4" -> "_IP_"
# - Sauvegarde train_anon.csv et test_anon.csv dans data/processed/
# =============================================================================
import time
from collections import Counter
from pathlib import Path

import pandas as pd
import spacy
from tqdm.auto import tqdm

# --- Chemins : marche que le notebook soit à la racine du repo ou dans un sous-dossier
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
OUT_DIR = DATA_DIR / "processed"
OUT_DIR.mkdir(exist_ok=True)

SAMPLE_SIZE = None   # None = tout le dataset (159 735 lignes)

# -----------------------------------------------------------------------------
# 1. Chargement du modèle spaCy anglais
#    tok2vec gardé : requis par le NER pour fonctionner correctement.
# -----------------------------------------------------------------------------
nlp = spacy.load(
    "en_core_web_sm",
    disable=["tagger", "parser", "attribute_ruler", "lemmatizer"],
)

# -----------------------------------------------------------------------------
# 2. Règles spaCy (EntityRuler) — placé AVANT le NER (prioritaire)
#    Couvre : emails, URLs, IPs, pseudos, et adresses postales
# -----------------------------------------------------------------------------
ruler = nlp.add_pipe("entity_ruler", before="ner")
ruler.add_patterns([
    {"label": "EMAIL",   "pattern": [{"LIKE_EMAIL": True}]},
    {"label": "URL",     "pattern": [{"LIKE_URL": True}]},
    {"label": "IP",      "pattern": [{"TEXT": {"REGEX": r"^(?:\d{1,3}\.){3}\d{1,3}$"}}]},
    {"label": "USER",    "pattern": [
        {"LOWER": {"IN": ["user", "user_talk"]}},
        {"LOWER": "talk", "OP": "?"},
        {"ORTH": ":"},
        {"IS_PUNCT": False, "IS_SPACE": False},
    ]},
    {"label": "USER",    "pattern": [{"TEXT": {"REGEX": r"^[A-Za-z][A-Za-z_]{2,}\d{2,}$"}}]},
    {"label": "ADDRESS", "pattern": [
        {"IS_DIGIT": True},
        {"IS_ALPHA": True, "OP": "+"},
        {"LOWER": {"IN": [
            "street", "st", "avenue", "ave", "road", "rd", "boulevard", "blvd",
            "drive", "dr", "lane", "ln", "court", "ct", "way", "place", "pl",
            "square", "sq", "terrace", "ter", "close", "grove", "park", "hill",
        ]}},
    ]},
    {"label": "ADDRESS", "pattern": [{"TEXT": {"REGEX": r"^\d{5}(-\d{4})?$"}}]},
])

# -----------------------------------------------------------------------------
# 3. Entités à masquer (RGPD — identification d'une personne physique)
#    Format underscore (_PERSON_) au lieu de crochets ([PERSON]) pour faciliter
#    le nettoyage de texte à l'étape 2 (pas de caractères spéciaux à échapper).
#    ORG retiré : ne permet pas d'identifier une personne physique (RGPD art. 4)
#    NORP, GPE, LOC gardés : signal essentiel pour identity_hate.
# -----------------------------------------------------------------------------
ENTITIES_TO_MASK = {"PERSON", "EMAIL", "URL", "IP", "USER", "ADDRESS"}

PROFANITY = {
    "fuck", "fucking", "fucker", "shit", "bitch", "ass", "asshole", "dick",
    "cunt", "cock", "cocksucker", "bastard", "damn", "suck", "sucks",
    "faggot", "fag", "nigga", "nigger", "whore", "slut", "idiot", "moron",
    "stupid", "gay", "retard", "penis", "pussy", "wanker", "twat",
}


def is_false_person(ent):
    """True si l'entité PERSON contient un mot grossier -> faux positif du NER."""
    return ent.label_ == "PERSON" and any(t.lower_ in PROFANITY for t in ent)


def anonymize_doc(doc):
    """Reconstruit le texte en remplaçant chaque entité à masquer par _LABEL_."""
    parts, masked, skipped, cursor = [], [], [], 0
    for ent in doc.ents:
        if ent.label_ not in ENTITIES_TO_MASK:
            continue
        if is_false_person(ent):
            skipped.append(ent.text)
            continue
        parts.append(doc.text[cursor:ent.start_char])
        parts.append(f"_{ent.label_}_")   # underscores : pas de caractères spéciaux
        masked.append(ent.label_)
        cursor = ent.end_char
    parts.append(doc.text[cursor:])
    return "".join(parts), masked, skipped


def anonymize_df(df, name):
    """Anonymise la colonne comment_text d'un DataFrame et affiche les stats."""
    texts = df["comment_text"].fillna("").astype(str).tolist()
    start = time.perf_counter()

    anon_texts, n_masked = [], []
    stats, skipped_all, persons = Counter(), Counter(), Counter()

    pipe = nlp.pipe(texts, batch_size=1000, n_process=1)
    for doc in tqdm(pipe, total=len(texts), desc=f"Anonymisation {name}"):
        text, masked, skipped = anonymize_doc(doc)
        anon_texts.append(text)
        n_masked.append(len(masked))
        stats.update(masked)
        skipped_all.update(skipped)
        persons.update(e.text for e in doc.ents if e.label_ == "PERSON" and not is_false_person(e))

    out = df.copy()
    out["comment_text_anon"] = anon_texts
    out["n_masked"] = n_masked

    print(f"=== {name} : {len(out)} lignes en {time.perf_counter() - start:.0f}s ===")
    print(f"Commentaires modifiés : {(out['n_masked'] > 0).mean():.1%}")
    print(f"Entités masquées      : {dict(stats.most_common())}")
    print(f"Faux PERSON ignorés   : {sum(skipped_all.values())}\n")
    return out, persons, skipped_all


# -----------------------------------------------------------------------------
# 4. Anonymisation de chaque fichier contenant du texte
# -----------------------------------------------------------------------------
for name in ["train", "test"]:
    df = pd.read_csv(DATA_DIR / f"{name}.csv")
    if SAMPLE_SIZE:
        df = df.sample(n=SAMPLE_SIZE, random_state=42)

    anon, persons, skipped = anonymize_df(df, name)

    anon["comment_text"] = anon["comment_text_anon"]
    anon = anon.drop(columns=["comment_text_anon"])
    anon.to_csv(OUT_DIR / f"{name}_anon.csv", index=False)
    print(f"-> {OUT_DIR / f'{name}_anon.csv'}\n")

    if name == "train":
        train, train_anon, train_persons, train_skipped = df, anon, persons, skipped

train_anon["comment_text_original"] = train["comment_text"]  # en mémoire seulement

# -----------------------------------------------------------------------------
# 5. Audit : les PERSON masqués les plus fréquents
# -----------------------------------------------------------------------------
print("\nTop 30 PERSON masqués :", train_persons.most_common(30))
print("\nFaux PERSON ignorés    :", train_skipped.most_common(15))

# -----------------------------------------------------------------------------
# 6. Comparaison avant / après
# -----------------------------------------------------------------------------
pd.set_option("display.max_colwidth", 150)
train_anon.loc[train_anon["n_masked"] > 0, ["comment_text_original", "comment_text"]].head(15)